## Testing Methods in the `IMAGETOOLS` Module

This Jupyter Notebook serves as a platform to rigorously test the various methods included in the Python module `IMAGETOOLS`.

This module is particularly valuable for researchers manipulating volumetric neuroimaging data:
performing morphological operations on binary masks, querying geometric attributes from the affine
matrix, cropping and restoring images, applying spatial transformations and converting between voxel
and millimeter coordinates, handling 4D images, and operating directly over 3D/4D arrays (mesh and
centroid extraction, probability maps, statistics, interpolation and region growing).

The tests are organized into the following sections for clarity and ease of navigation:

### Table of Contents

---

* 🔷 1. [`MorphologicalOperations Class`](#morphops-class). Main class to perform morphological operations on binary arrays
    * 🔸 1.1. [`create_structuring_element`](#morphops-create_structuring_element). Create a structuring element for morphological operations
    * 🔸 1.2. [`erode`](#morphops-erode). Perform binary erosion to shrink objects and remove small noise
    * 🔸 1.3. [`dilate`](#morphops-dilate). Perform binary dilation to expand objects and fill small gaps
    * 🔸 1.4. [`dilate_mm`](#morphops-dilate_mm). Binarize a 3D array and dilate by a given distance in millimeters
    * 🔸 1.5. [`opening`](#morphops-opening). Perform morphological opening (erosion followed by dilation)
    * 🔸 1.6. [`closing`](#morphops-closing). Perform morphological closing (dilation followed by erosion)
    * 🔸 1.7. [`fill_holes`](#morphops-fill_holes). Fill holes in binary objects
    * 🔸 1.8. [`detect_communities`](#morphops-detect_communities). Detect connected components (communities) in a binary array
    * 🔸 1.9. [`remove_small_objects`](#morphops-remove_small_objects). Remove connected components smaller than the specified size
    * 🔸 1.10. [`gradient`](#morphops-gradient). Morphological gradient (dilation - erosion) to highlight object boundaries
    * 🔸 1.11. [`tophat`](#morphops-tophat). White top-hat transform (original - opening) to extract small bright structures
    * 🔸 1.12. [`blackhat`](#morphops-blackhat). Black top-hat transform (closing - original) to extract small dark structures
    * 🔸 1.13. [`quick_morphology`](#morphops-quick_morphology). Quick access to the morphological operations without creating a class instance

* 🔷 2. [`Attributes from Images`](#image-attributes). Methods to get attributes from the images
    * 🔸 2.1. [`get_voxel_size`](#image-get_voxel_size). Compute the voxel dimensions from a NIfTI affine matrix
    * 🔸 2.2. [`get_voxel_volume`](#image-get_voxel_volume). Compute the voxel volume from an affine matrix
    * 🔸 2.3. [`get_center`](#image-get_center). Compute the center of the image from a NIfTI affine matrix
    * 🔸 2.4. [`get_rotation_matrix`](#image-get_rotation_matrix). Extract the normalized rotation matrix from an affine matrix
    * 🔸 2.5. [`get_vox_neighbors`](#image-get_vox_neighbors). Get the neighborhood coordinates for a voxel

* 🔷 3. [`Operations over Images`](#image-operations). Methods to operate over images (e.g. crop)
    * 🔸 3.1. [`crop_image_from_mask`](#image-crop_image_from_mask). Crop an image using a mask to the minimum bounding box containing the specified structures
    * 🔸 3.2. [`cropped_to_native`](#image-cropped_to_native). Restore a cropped image to the dimensions of a reference native image

* 🔷 4. [`Transformations and Changes of Space`](#image-transformations). Methods to apply transformations or changes of space
    * 🔸 4.1. [`apply_multi_transf`](#image-apply_multi_transf). Apply an ANTs transformation to an image with support for multiple transform types
    * 🔸 4.2. [`vox2mm`](#image-vox2mm). Convert voxel coordinates to millimeter coordinates using an affine matrix
    * 🔸 4.3. [`mm2vox`](#image-mm2vox). Convert millimeter coordinates to voxel coordinates using an affine matrix

* 🔷 5. [`4D Images`](#image-4d). Methods to work with 4D images
    * 🔸 5.1. [`merge_to_4d`](#image-merge_to_4d). Merge multiple 3D or 4D NIfTI files into a single 4D NIfTI file
    * 🔸 5.2. [`create_spams`](#image-create_spams). Create SPAM probability maps from multiple parcellation images
    * 🔸 5.3. [`spams2maxprob`](#image-spams2maxprob). Convert SPAM probability maps to a maximum probability parcellation
    * 🔸 5.4. [`simulate_image`](#image-simulate_image). Generate a simulated image with random values at non-zero voxel positions
    * 🔸 5.5. [`delete_volumes_from_4D_images`](#image-delete_volumes_from_4D_images). Remove specific volumes from a 4D neuroimaging file

* 🔷 6. [`Operations from 3D or 4D Arrays`](#image-arrays). Methods to perform operations from 3D or 4D arrays
    * 🔸 6.1. [`extract_mesh_from_volume`](#image-extract_mesh_from_volume). Extract a surface mesh from a 3D volume using the marching cubes algorithm
    * 🔸 6.2. [`extract_centroid_from_volume`](#image-extract_centroid_from_volume). Extract the centroid and the voxel count from a 3D binary volume
    * 🔸 6.3. [`create_spams_from_volume`](#image-create_spams_from_volume). Create SPAMs (Spatial Probability Maps) from individual parcellation volumes
    * 🔸 6.4. [`spams2maxprob_from_volume`](#image-spams2maxprob_from_volume). Convert SPAMs (Spatial Probability Maps) to a maximum probability parcellation volume
    * 🔸 6.5. [`compute_statistics_at_nonzero_voxels`](#image-compute_statistics_at_nonzero_voxels). Compute a given statistic from a data array at the positions where the mask array is non-zero
    * 🔸 6.6. [`interpolate`](#image-interpolate). Interpolate 3D or 4D scalar data at specified voxel coordinates using regular grid interpolation
    * 🔸 6.7. [`region_growing`](#image-region_growing). Fill gaps in a parcellation using a region growing algorithm
    * 🔸 6.8. [`simulate_array`](#image-simulate_array). Generate a simulated array with random values at non-zero voxel positions
    * 🔸 6.9. [`delete_volumes_from_4D_array`](#image-delete_volumes_from_4D_array). Remove specific volumes from a 4D array
    * 🔸 6.10. [`dilate_mm`](#image-dilate_mm). Binarize a 3D array and dilate by a given distance in millimeters

---

---
## <a id="morphops-class"></a>1. MorphologicalOperations Class
Main class to perform morphological operations on binary arrays.

### 1.1 `create_structuring_element`
<a id="morphops-create_structuring_element"></a>

Create a structuring element for morphological operations.

In [ ]:
########################### Testing create_structuring_element ###########################
import clabtoolkit.imagetools as cltimg
import numpy as np

morph = cltimg.MorphologicalOperations()

print("Test 1: Creating the default structuring element (3 x 3 x 3 cube)...")
cube = morph.create_structuring_element()
print(f"  Shape: {cube.shape}, active elements: {cube.sum()}")
print("-------------------------------")
print(" ")

print("Test 2: Creating a 3D ball of size 7 and a 2D disk of size 5...")
ball = morph.create_structuring_element(shape="ball", size=7, dimensions=3)
disk = morph.create_structuring_element(shape="disk", size=5, dimensions=2)
print(f"  Ball shape: {ball.shape}, active elements: {ball.sum()}")
print(f"  Disk shape: {disk.shape}:")
print(disk.astype(int))
print("-------------------------------")
print(" ")

print("Test 3: Creating 2D and 3D cross-shaped elements (face connectivity only)...")
cross2d = morph.create_structuring_element(shape="cross", dimensions=2)
cross3d = morph.create_structuring_element(shape="cross", dimensions=3)
print(f"  2D cross:\n{cross2d.astype(int)}")
print(f"  3D cross active elements: {cross3d.sum()} (center + 6 face neighbors)")
print("-------------------------------")
print(" ")

print("Test 4: Requesting an unsupported shape...")
try:
    morph.create_structuring_element(shape="pyramid")
except ValueError as e:
    print(f"  ValueError: {e}")
print("-------------------------------")


### 1.2 `erode`
<a id="morphops-erode"></a>

Perform binary erosion to shrink objects and remove small noise.

In [ ]:
########################### Testing erode ###########################
import clabtoolkit.imagetools as cltimg
import numpy as np
import matplotlib.pyplot as plt

morph = cltimg.MorphologicalOperations()

# 2D toy mask: a square with a small hole, a rectangle and two isolated noisy pixels
img2d = np.zeros((40, 40), dtype=bool)
img2d[5:20, 5:20] = True       # Square (15 x 15)
img2d[10:13, 10:13] = False    # 3 x 3 hole inside the square
img2d[25:35, 22:36] = True     # Rectangle (10 x 14)
img2d[30, 5] = True            # Noisy pixel
img2d[3, 35] = True            # Noisy pixel

# 3D toy mask: a sphere of radius 10 voxels centered in a 40 x 40 x 40 volume
zz, yy, xx = np.ogrid[:40, :40, :40]
sphere = (xx - 20) ** 2 + (yy - 20) ** 2 + (zz - 20) ** 2 <= 10**2

def show(panels):
    """Display a list of (title, 2D array) panels side by side."""
    fig, axes = plt.subplots(1, len(panels), figsize=(3 * len(panels), 3))
    for ax, (title, im) in zip(axes, panels):
        ax.imshow(im, cmap="gray", interpolation="nearest")
        ax.set_title(title, fontsize=10)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

print("Test 1: Eroding the 2D mask with the default 3 x 3 square. The noisy pixels disappear...")
eroded = morph.erode(img2d)
print(f"  Pixels before: {img2d.sum()}, after: {eroded.sum()}")
print("-------------------------------")
print(" ")

print("Test 2: Eroding the 3D sphere with 1 and 3 iterations...")
for it in [1, 3]:
    er = morph.erode(sphere, iterations=it)
    print(f"  iterations={it}: {sphere.sum()} -> {er.sum()} voxels")
print("-------------------------------")
print(" ")

print("Test 3: Eroding with a cross-shaped element (removes less than the square)...")
cross = morph.create_structuring_element("cross", dimensions=2)
eroded_cross = morph.erode(img2d, structure=cross)
print(f"  Square element: {eroded.sum()} pixels, cross element: {eroded_cross.sum()} pixels")
print("-------------------------------")

show([("Original", img2d), ("Eroded (square)", eroded), ("Eroded (cross)", eroded_cross)])


### 1.3 `dilate`
<a id="morphops-dilate"></a>

Perform binary dilation to expand objects and fill small gaps.

In [ ]:
########################### Testing dilate ###########################
import clabtoolkit.imagetools as cltimg
import numpy as np
import matplotlib.pyplot as plt

morph = cltimg.MorphologicalOperations()

# 2D toy mask: a square with a small hole, a rectangle and two isolated noisy pixels
img2d = np.zeros((40, 40), dtype=bool)
img2d[5:20, 5:20] = True       # Square (15 x 15)
img2d[10:13, 10:13] = False    # 3 x 3 hole inside the square
img2d[25:35, 22:36] = True     # Rectangle (10 x 14)
img2d[30, 5] = True            # Noisy pixel
img2d[3, 35] = True            # Noisy pixel

# 3D toy mask: a sphere of radius 10 voxels centered in a 40 x 40 x 40 volume
zz, yy, xx = np.ogrid[:40, :40, :40]
sphere = (xx - 20) ** 2 + (yy - 20) ** 2 + (zz - 20) ** 2 <= 10**2

def show(panels):
    """Display a list of (title, 2D array) panels side by side."""
    fig, axes = plt.subplots(1, len(panels), figsize=(3 * len(panels), 3))
    for ax, (title, im) in zip(axes, panels):
        ax.imshow(im, cmap="gray", interpolation="nearest")
        ax.set_title(title, fontsize=10)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

print("Test 1: Dilating the 2D mask with the default 3 x 3 square...")
dilated = morph.dilate(img2d)
print(f"  Pixels before: {img2d.sum()}, after: {dilated.sum()}")
print("-------------------------------")
print(" ")

print("Test 2: Dilating the 3D sphere with 2 iterations...")
dil3d = morph.dilate(sphere, iterations=2)
print(f"  Voxels: {sphere.sum()} -> {dil3d.sum()}")
print("-------------------------------")
print(" ")

print("Test 3: Dilating with a disk of size 7 (rounder, larger expansion)...")
disk = morph.create_structuring_element("disk", size=7, dimensions=2)
dilated_disk = morph.dilate(img2d, structure=disk)
print(f"  Pixels after dilation with the disk: {dilated_disk.sum()}")
print("-------------------------------")

show([("Original", img2d), ("Dilated (3x3)", dilated), ("Dilated (disk 7)", dilated_disk)])


### 1.4 `dilate_mm`
<a id="morphops-dilate_mm"></a>

Binarize a 3D array and dilate by a given distance in millimeters.

In [ ]:
########################### Testing dilate_mm ###########################
import clabtoolkit.imagetools as cltimg
import numpy as np

morph = cltimg.MorphologicalOperations()

# A single active voxel in the center of the volume
seed = np.zeros((21, 21, 21), dtype=np.uint8)
seed[10, 10, 10] = 1

def extent(mask):
    """Number of voxels covered along each axis."""
    idx = np.argwhere(mask)
    return tuple(int(v) for v in idx.max(axis=0) - idx.min(axis=0) + 1)

print("Test 1: Dilating a single voxel by 2 mm with 1 mm isotropic voxels (cube)...")
affine_iso = np.diag([1.0, 1.0, 1.0, 1.0])
dil = morph.dilate_mm(seed, affine_iso, shape="cube", dilation_mm=2)
print(f"  Active voxels: {dil.sum()}, extent (voxels): {extent(dil)}")
print("-------------------------------")
print(" ")

print("Test 2: The same dilation with a ball-shaped element...")
dil_ball = morph.dilate_mm(seed, affine_iso, shape="ball", dilation_mm=2)
print(f"  Active voxels: {dil_ball.sum()}, extent (voxels): {extent(dil_ball)}")
print("-------------------------------")
print(" ")

print("Test 3: Anisotropic voxels (1 x 1 x 3 mm). The radius in voxels adapts to each axis...")
affine_aniso = np.diag([1.0, 1.0, 3.0, 1.0])
dil_aniso = morph.dilate_mm(seed, affine_aniso, shape="ball", dilation_mm=3)
print(f"  Active voxels: {dil_aniso.sum()}, extent (voxels): {extent(dil_aniso)}")
print("-------------------------------")


### 1.5 `opening`
<a id="morphops-opening"></a>

Perform morphological opening (erosion followed by dilation).

In [ ]:
########################### Testing opening ###########################
import clabtoolkit.imagetools as cltimg
import numpy as np
import matplotlib.pyplot as plt

morph = cltimg.MorphologicalOperations()

# 2D toy mask: a square with a small hole, a rectangle and two isolated noisy pixels
img2d = np.zeros((40, 40), dtype=bool)
img2d[5:20, 5:20] = True       # Square (15 x 15)
img2d[10:13, 10:13] = False    # 3 x 3 hole inside the square
img2d[25:35, 22:36] = True     # Rectangle (10 x 14)
img2d[30, 5] = True            # Noisy pixel
img2d[3, 35] = True            # Noisy pixel

# 3D toy mask: a sphere of radius 10 voxels centered in a 40 x 40 x 40 volume
zz, yy, xx = np.ogrid[:40, :40, :40]
sphere = (xx - 20) ** 2 + (yy - 20) ** 2 + (zz - 20) ** 2 <= 10**2

def show(panels):
    """Display a list of (title, 2D array) panels side by side."""
    fig, axes = plt.subplots(1, len(panels), figsize=(3 * len(panels), 3))
    for ax, (title, im) in zip(axes, panels):
        ax.imshow(im, cmap="gray", interpolation="nearest")
        ax.set_title(title, fontsize=10)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

print("Test 1: Opening the 2D mask removes the isolated noisy pixels but keeps the objects...")
opened = morph.opening(img2d)
print(f"  Pixels before: {img2d.sum()}, after: {opened.sum()}")
print("-------------------------------")
print(" ")

print("Test 2: Opening a 3D sphere with a thin 1-voxel spur attached to it...")
sphere_spur = sphere.copy()
sphere_spur[20, 20, 30:39] = True
opened3d = morph.opening(sphere_spur)
print(f"  Spur voxels before: {sphere_spur[20, 20, 31:39].sum()}, after: {opened3d[20, 20, 31:39].sum()}")
print("-------------------------------")
print(" ")

print("Test 3: Opening with a larger disk removes the corners of the objects...")
disk = morph.create_structuring_element("disk", size=7, dimensions=2)
opened_disk = morph.opening(img2d, structure=disk)
print(f"  Pixels after opening with the disk: {opened_disk.sum()}")
print("-------------------------------")

show([("Original", img2d), ("Opened (3x3)", opened), ("Opened (disk 7)", opened_disk)])


### 1.6 `closing`
<a id="morphops-closing"></a>

Perform morphological closing (dilation followed by erosion).

In [ ]:
########################### Testing closing ###########################
import clabtoolkit.imagetools as cltimg
import numpy as np
import matplotlib.pyplot as plt

morph = cltimg.MorphologicalOperations()

# 2D toy mask: a square with a small hole, a rectangle and two isolated noisy pixels
img2d = np.zeros((40, 40), dtype=bool)
img2d[5:20, 5:20] = True       # Square (15 x 15)
img2d[10:13, 10:13] = False    # 3 x 3 hole inside the square
img2d[25:35, 22:36] = True     # Rectangle (10 x 14)
img2d[30, 5] = True            # Noisy pixel
img2d[3, 35] = True            # Noisy pixel

# 3D toy mask: a sphere of radius 10 voxels centered in a 40 x 40 x 40 volume
zz, yy, xx = np.ogrid[:40, :40, :40]
sphere = (xx - 20) ** 2 + (yy - 20) ** 2 + (zz - 20) ** 2 <= 10**2

def show(panels):
    """Display a list of (title, 2D array) panels side by side."""
    fig, axes = plt.subplots(1, len(panels), figsize=(3 * len(panels), 3))
    for ax, (title, im) in zip(axes, panels):
        ax.imshow(im, cmap="gray", interpolation="nearest")
        ax.set_title(title, fontsize=10)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

print("Test 1: Closing the 2D mask with 1 and 2 iterations. The 3 x 3 hole needs 2 iterations...")
closed1 = morph.closing(img2d, iterations=1)
closed2 = morph.closing(img2d, iterations=2)
print(f"  Hole pixels left: iterations=1 -> {(~closed1[10:13, 10:13]).sum()}, iterations=2 -> {(~closed2[10:13, 10:13]).sum()}")
print("-------------------------------")
print(" ")

print("Test 2: Closing bridges a 1-voxel gap between two 3D blocks...")
blocks = np.zeros((20, 20, 20), dtype=bool)
blocks[5:15, 5:15, 2:9] = True
blocks[5:15, 5:15, 10:17] = True         # Gap at z = 9
closed_blocks = morph.closing(blocks)
print(f"  Voxels in the gap before: {blocks[5:15, 5:15, 9].sum()}, after: {closed_blocks[5:15, 5:15, 9].sum()}")
print("-------------------------------")
print(" ")

print("Test 3: Closing with a cross-shaped element...")
cross = morph.create_structuring_element("cross", dimensions=2)
closed_cross = morph.closing(img2d, structure=cross, iterations=2)
print(f"  Pixels: {img2d.sum()} -> {closed_cross.sum()}")
print("-------------------------------")

show([("Original", img2d), ("Closed (1 it.)", closed1), ("Closed (2 it.)", closed2)])


### 1.7 `fill_holes`
<a id="morphops-fill_holes"></a>

Fill holes in binary objects.

In [ ]:
########################### Testing fill_holes ###########################
import clabtoolkit.imagetools as cltimg
import numpy as np
import matplotlib.pyplot as plt

morph = cltimg.MorphologicalOperations()

# 2D toy mask: a square with a small hole, a rectangle and two isolated noisy pixels
img2d = np.zeros((40, 40), dtype=bool)
img2d[5:20, 5:20] = True       # Square (15 x 15)
img2d[10:13, 10:13] = False    # 3 x 3 hole inside the square
img2d[25:35, 22:36] = True     # Rectangle (10 x 14)
img2d[30, 5] = True            # Noisy pixel
img2d[3, 35] = True            # Noisy pixel

# 3D toy mask: a sphere of radius 10 voxels centered in a 40 x 40 x 40 volume
zz, yy, xx = np.ogrid[:40, :40, :40]
sphere = (xx - 20) ** 2 + (yy - 20) ** 2 + (zz - 20) ** 2 <= 10**2

def show(panels):
    """Display a list of (title, 2D array) panels side by side."""
    fig, axes = plt.subplots(1, len(panels), figsize=(3 * len(panels), 3))
    for ax, (title, im) in zip(axes, panels):
        ax.imshow(im, cmap="gray", interpolation="nearest")
        ax.set_title(title, fontsize=10)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

print("Test 1: Filling the hole of the 2D square...")
filled = morph.fill_holes(img2d)
print(f"  Pixels before: {img2d.sum()}, after: {filled.sum()} (+{filled.sum() - img2d.sum()})")
print("-------------------------------")
print(" ")

print("Test 2: Filling a hollow 3D sphere (a 2-voxel thick shell)...")
shell = sphere & ~morph.erode(sphere, iterations=2)
filled_shell = morph.fill_holes(shell)
print(f"  Shell voxels: {shell.sum()}, filled voxels: {filled_shell.sum()}, solid sphere voxels: {sphere.sum()}")
print("-------------------------------")
print(" ")

print("Test 3: A hole that touches the border is not a hole (it is connected to the background)...")
open_ring = img2d.copy()
open_ring[11, 5:10] = False                # Cut a channel from the hole to the outside
filled_open = morph.fill_holes(open_ring)
print(f"  Pixels added: {filled_open.sum() - open_ring.sum()}")
print("-------------------------------")

show([("Original", img2d), ("Filled", filled), ("Open hole (unchanged)", filled_open)])


### 1.8 `detect_communities`
<a id="morphops-detect_communities"></a>

Detect connected components (communities) in a binary array.

In [ ]:
########################### Testing detect_communities ###########################
import clabtoolkit.imagetools as cltimg
import numpy as np
import matplotlib.pyplot as plt

morph = cltimg.MorphologicalOperations()

# 2D toy mask: a square with a small hole, a rectangle and two isolated noisy pixels
img2d = np.zeros((40, 40), dtype=bool)
img2d[5:20, 5:20] = True       # Square (15 x 15)
img2d[10:13, 10:13] = False    # 3 x 3 hole inside the square
img2d[25:35, 22:36] = True     # Rectangle (10 x 14)
img2d[30, 5] = True            # Noisy pixel
img2d[3, 35] = True            # Noisy pixel

# 3D toy mask: a sphere of radius 10 voxels centered in a 40 x 40 x 40 volume
zz, yy, xx = np.ogrid[:40, :40, :40]
sphere = (xx - 20) ** 2 + (yy - 20) ** 2 + (zz - 20) ** 2 <= 10**2

def show(panels):
    """Display a list of (title, 2D array) panels side by side."""
    fig, axes = plt.subplots(1, len(panels), figsize=(3 * len(panels), 3))
    for ax, (title, im) in zip(axes, panels):
        ax.imshow(im, cmap="gray", interpolation="nearest")
        ax.set_title(title, fontsize=10)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

print("Test 1: Detecting the connected components of the 2D mask...")
labeled, n = morph.detect_communities(img2d)
sizes = np.bincount(labeled.ravel())[1:]
print(f"  Components: {n}, sizes: {sizes}")
print("-------------------------------")
print(" ")

print("Test 2: Connectivity matters. Two diagonal pixels are one component with the square element and two with the cross...")
diag = np.zeros((5, 5), dtype=bool)
diag[1, 1] = diag[2, 2] = True
_, n_square = morph.detect_communities(diag)
_, n_cross = morph.detect_communities(diag, structure=morph.create_structuring_element("cross", dimensions=2))
print(f"  Square (8-connectivity): {n_square}, cross (4-connectivity): {n_cross}")
print("-------------------------------")
print(" ")

print("Test 3: Detecting three separated spheres in a 3D volume...")
three = np.zeros((60, 30, 30), dtype=bool)
zz, yy, xx = np.ogrid[:60, :30, :30]
for cx in [10, 30, 50]:
    three |= (zz - cx) ** 2 + (yy - 15) ** 2 + (xx - 15) ** 2 <= 6**2
labeled3d, n3d = morph.detect_communities(three)
print(f"  Components: {n3d}, voxels per component: {np.bincount(labeled3d.ravel())[1:]}")
print("-------------------------------")

show([("Original", img2d), ("Labeled components", labeled)])


### 1.9 `remove_small_objects`
<a id="morphops-remove_small_objects"></a>

Remove connected components smaller than the specified size.

In [ ]:
########################### Testing remove_small_objects ###########################
import clabtoolkit.imagetools as cltimg
import numpy as np
import matplotlib.pyplot as plt

morph = cltimg.MorphologicalOperations()

# 2D toy mask: a square with a small hole, a rectangle and two isolated noisy pixels
img2d = np.zeros((40, 40), dtype=bool)
img2d[5:20, 5:20] = True       # Square (15 x 15)
img2d[10:13, 10:13] = False    # 3 x 3 hole inside the square
img2d[25:35, 22:36] = True     # Rectangle (10 x 14)
img2d[30, 5] = True            # Noisy pixel
img2d[3, 35] = True            # Noisy pixel

# 3D toy mask: a sphere of radius 10 voxels centered in a 40 x 40 x 40 volume
zz, yy, xx = np.ogrid[:40, :40, :40]
sphere = (xx - 20) ** 2 + (yy - 20) ** 2 + (zz - 20) ** 2 <= 10**2

def show(panels):
    """Display a list of (title, 2D array) panels side by side."""
    fig, axes = plt.subplots(1, len(panels), figsize=(3 * len(panels), 3))
    for ax, (title, im) in zip(axes, panels):
        ax.imshow(im, cmap="gray", interpolation="nearest")
        ax.set_title(title, fontsize=10)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

print("Test 1: Removing the noisy pixels (objects smaller than 5 pixels)...")
clean = morph.remove_small_objects(img2d, min_size=5)
print(f"  Components before: {morph.detect_communities(img2d)[1]}, after: {morph.detect_communities(clean)[1]}")
print("-------------------------------")
print(" ")

print("Test 2: Keeping only the large objects (at least 150 pixels). The rectangle (140 px) is removed...")
large = morph.remove_small_objects(img2d, min_size=150)
print(f"  Pixels kept: {large.sum()}")
print("-------------------------------")
print(" ")

print("Test 3: Cleaning a noisy 3D sphere (random scattered voxels around it)...")
rng = np.random.default_rng(0)
noisy = sphere | (rng.random(sphere.shape) > 0.995)
clean3d = morph.remove_small_objects(noisy, min_size=50)
print(f"  Voxels: noisy={noisy.sum()}, cleaned={clean3d.sum()}, sphere={sphere.sum()}")
print("-------------------------------")

show([("Original", img2d), ("min_size=5", clean), ("min_size=150", large)])


### 1.10 `gradient`
<a id="morphops-gradient"></a>

Morphological gradient (dilation - erosion) to highlight object boundaries.

In [ ]:
########################### Testing gradient ###########################
import clabtoolkit.imagetools as cltimg
import numpy as np
import matplotlib.pyplot as plt

morph = cltimg.MorphologicalOperations()

# 2D toy mask: a square with a small hole, a rectangle and two isolated noisy pixels
img2d = np.zeros((40, 40), dtype=bool)
img2d[5:20, 5:20] = True       # Square (15 x 15)
img2d[10:13, 10:13] = False    # 3 x 3 hole inside the square
img2d[25:35, 22:36] = True     # Rectangle (10 x 14)
img2d[30, 5] = True            # Noisy pixel
img2d[3, 35] = True            # Noisy pixel

# 3D toy mask: a sphere of radius 10 voxels centered in a 40 x 40 x 40 volume
zz, yy, xx = np.ogrid[:40, :40, :40]
sphere = (xx - 20) ** 2 + (yy - 20) ** 2 + (zz - 20) ** 2 <= 10**2

def show(panels):
    """Display a list of (title, 2D array) panels side by side."""
    fig, axes = plt.subplots(1, len(panels), figsize=(3 * len(panels), 3))
    for ax, (title, im) in zip(axes, panels):
        ax.imshow(im, cmap="gray", interpolation="nearest")
        ax.set_title(title, fontsize=10)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

print("Test 1: Extracting the boundaries of the 2D objects...")
edges = morph.gradient(img2d)
print(f"  Boundary pixels: {edges.sum()}")
print("-------------------------------")
print(" ")

print("Test 2: Using a cross-shaped element produces thinner boundaries...")
cross = morph.create_structuring_element("cross", dimensions=2)
edges_cross = morph.gradient(img2d, structure=cross)
print(f"  Boundary pixels: square={edges.sum()}, cross={edges_cross.sum()}")
print("-------------------------------")
print(" ")

print("Test 3: Extracting the surface shell of the 3D sphere...")
shell = morph.gradient(sphere)
print(f"  Sphere voxels: {sphere.sum()}, shell voxels: {shell.sum()}")
print("-------------------------------")

show([("Original", img2d), ("Gradient (square)", edges), ("Gradient (cross)", edges_cross)])


### 1.11 `tophat`
<a id="morphops-tophat"></a>

White top-hat transform (original - opening) to extract small bright structures.

In [ ]:
########################### Testing tophat ###########################
import clabtoolkit.imagetools as cltimg
import numpy as np
import matplotlib.pyplot as plt

morph = cltimg.MorphologicalOperations()

# 2D toy mask: a square with a small hole, a rectangle and two isolated noisy pixels
img2d = np.zeros((40, 40), dtype=bool)
img2d[5:20, 5:20] = True       # Square (15 x 15)
img2d[10:13, 10:13] = False    # 3 x 3 hole inside the square
img2d[25:35, 22:36] = True     # Rectangle (10 x 14)
img2d[30, 5] = True            # Noisy pixel
img2d[3, 35] = True            # Noisy pixel

# 3D toy mask: a sphere of radius 10 voxels centered in a 40 x 40 x 40 volume
zz, yy, xx = np.ogrid[:40, :40, :40]
sphere = (xx - 20) ** 2 + (yy - 20) ** 2 + (zz - 20) ** 2 <= 10**2

def show(panels):
    """Display a list of (title, 2D array) panels side by side."""
    fig, axes = plt.subplots(1, len(panels), figsize=(3 * len(panels), 3))
    for ax, (title, im) in zip(axes, panels):
        ax.imshow(im, cmap="gray", interpolation="nearest")
        ax.set_title(title, fontsize=10)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

# Add a thin 1-pixel line attached to the rectangle
img_line = img2d.copy()
img_line[30, 36:39] = True

print("Test 1: The white top-hat extracts the structures thinner than the element (noise and the line)...")
th = morph.tophat(img_line)
print(f"  Pixels extracted: {th.sum()} at rows/cols: {np.argwhere(th).tolist()}")
print("-------------------------------")
print(" ")

print("Test 2: A larger element (disk of size 7) also extracts the corners of the objects...")
disk = morph.create_structuring_element("disk", size=7, dimensions=2)
th_disk = morph.tophat(img_line, structure=disk)
print(f"  Pixels extracted: {th_disk.sum()}")
print("-------------------------------")
print(" ")

print("Test 3: Top-hat on the 3D sphere with a spur. Only the spur is extracted...")
sphere_spur = sphere.copy()
sphere_spur[20, 20, 31:39] = True
th3d = morph.tophat(sphere_spur)
print(f"  Voxels extracted: {th3d.sum()}")
print("-------------------------------")

show([("Original", img_line), ("Top-hat (3x3)", th), ("Top-hat (disk 7)", th_disk)])


### 1.12 `blackhat`
<a id="morphops-blackhat"></a>

Black top-hat transform (closing - original) to extract small dark structures.

In [ ]:
########################### Testing blackhat ###########################
import clabtoolkit.imagetools as cltimg
import numpy as np
import matplotlib.pyplot as plt

morph = cltimg.MorphologicalOperations()

# 2D toy mask: a square with a small hole, a rectangle and two isolated noisy pixels
img2d = np.zeros((40, 40), dtype=bool)
img2d[5:20, 5:20] = True       # Square (15 x 15)
img2d[10:13, 10:13] = False    # 3 x 3 hole inside the square
img2d[25:35, 22:36] = True     # Rectangle (10 x 14)
img2d[30, 5] = True            # Noisy pixel
img2d[3, 35] = True            # Noisy pixel

# 3D toy mask: a sphere of radius 10 voxels centered in a 40 x 40 x 40 volume
zz, yy, xx = np.ogrid[:40, :40, :40]
sphere = (xx - 20) ** 2 + (yy - 20) ** 2 + (zz - 20) ** 2 <= 10**2

def show(panels):
    """Display a list of (title, 2D array) panels side by side."""
    fig, axes = plt.subplots(1, len(panels), figsize=(3 * len(panels), 3))
    for ax, (title, im) in zip(axes, panels):
        ax.imshow(im, cmap="gray", interpolation="nearest")
        ax.set_title(title, fontsize=10)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

# Add a 1-pixel crack to the rectangle and a single-pixel hole to the square
img_dark = img2d.copy()
img_dark[25:35, 29] = False
img_dark[16, 16] = False

print("Test 1: The black top-hat extracts the small dark structures (the crack and the 1-pixel hole)...")
bh = morph.blackhat(img_dark)
print(f"  Pixels extracted: {bh.sum()}")
print("-------------------------------")
print(" ")

print("Test 2: With a larger element (disk of size 7) the 3 x 3 hole is also extracted...")
disk = morph.create_structuring_element("disk", size=7, dimensions=2)
bh_disk = morph.blackhat(img_dark, structure=disk)
print(f"  Pixels extracted: {bh_disk.sum()}, 3 x 3 hole included: {bh_disk[10:13, 10:13].all()}")
print("-------------------------------")
print(" ")

print("Test 3: Black top-hat on a 3D sphere with a small internal cavity...")
cavity = sphere.copy()
cavity[19:21, 19:21, 19:21] = False
bh3d = morph.blackhat(cavity)
print(f"  Cavity voxels: {(sphere & ~cavity).sum()}, extracted: {bh3d.sum()}")
print("-------------------------------")

show([("Original", img_dark), ("Black-hat (3x3)", bh), ("Black-hat (disk 7)", bh_disk)])


### 1.13 `quick_morphology`
<a id="morphops-quick_morphology"></a>

Quick access to the morphological operations without creating a class instance.

In [ ]:
########################### Testing quick_morphology ###########################
import clabtoolkit.imagetools as cltimg
import numpy as np
import matplotlib.pyplot as plt

morph = cltimg.MorphologicalOperations()

# 2D toy mask: a square with a small hole, a rectangle and two isolated noisy pixels
img2d = np.zeros((40, 40), dtype=bool)
img2d[5:20, 5:20] = True       # Square (15 x 15)
img2d[10:13, 10:13] = False    # 3 x 3 hole inside the square
img2d[25:35, 22:36] = True     # Rectangle (10 x 14)
img2d[30, 5] = True            # Noisy pixel
img2d[3, 35] = True            # Noisy pixel

# 3D toy mask: a sphere of radius 10 voxels centered in a 40 x 40 x 40 volume
zz, yy, xx = np.ogrid[:40, :40, :40]
sphere = (xx - 20) ** 2 + (yy - 20) ** 2 + (zz - 20) ** 2 <= 10**2

print("Test 1: Quick erosion with 2 iterations (no class instance needed)...")
er = cltimg.quick_morphology(img2d, "erode", iterations=2)
print(f"  Pixels: {img2d.sum()} -> {er.sum()}")
print("-------------------------------")
print(" ")

print("Test 2: Chaining quick operations: remove small objects and then fill holes...")
clean = cltimg.quick_morphology(img2d, "remove_small", min_size=5)
clean = cltimg.quick_morphology(clean, "fill_holes")
print(f"  Components: {morph.detect_communities(clean)[1]}, pixels: {clean.sum()}")
print("-------------------------------")
print(" ")

print("Test 3: Quick closing of a 3D sphere with a small internal cavity, using a ball-shaped element...")
cavity = sphere.copy()
cavity[19:21, 19:21, 19:21] = False
ball = morph.create_structuring_element("ball", size=5, dimensions=3)
closed = cltimg.quick_morphology(cavity, "closing", structure=ball)
print(f"  Voxels: {cavity.sum()} -> {closed.sum()} (solid sphere: {sphere.sum()})")
print("-------------------------------")
print(" ")

print("Test 4: Requesting an unsupported operation...")
try:
    cltimg.quick_morphology(img2d, "skeletonize")
except ValueError as e:
    print(f"  ValueError: {e}")
print("-------------------------------")


---
## <a id="image-attributes"></a>2. Attributes from Images
Methods to get attributes from the images.

### 2.1 `get_voxel_size`
<a id="image-get_voxel_size"></a>

Compute the voxel dimensions from a NIfTI affine matrix.

In [ ]:
########################### Testing get_voxel_size ###########################
import clabtoolkit.imagetools as cltimg
import numpy as np

# MNI152 1 mm affine (radiological flip on X, origin at the anterior commissure)
affine_mni = np.array([[-1.0, 0.0, 0.0,   90.0],
                       [ 0.0, 1.0, 0.0, -126.0],
                       [ 0.0, 0.0, 1.0,  -72.0],
                       [ 0.0, 0.0, 0.0,    1.0]])

# Typical DWI affine with anisotropic voxels (2 x 2 x 2.5 mm)
affine_dwi = np.diag([2.0, 2.0, 2.5, 1.0])
affine_dwi[:3, 3] = [-96.0, -96.0, -60.0]

# Oblique acquisition: 15 degrees rotation around Z with 2 mm isotropic voxels
theta = np.deg2rad(15)
rot_z = np.array([[np.cos(theta), -np.sin(theta), 0.0],
                  [np.sin(theta),  np.cos(theta), 0.0],
                  [0.0,            0.0,           1.0]])
affine_obl = np.eye(4)
affine_obl[:3, :3] = rot_z @ np.diag([2.0, 2.0, 2.0])
affine_obl[:3, 3] = [-80.0, -100.0, -50.0]

print("Test 1: Voxel size of the MNI152 1 mm template (the X flip does not change the size)...")
print(f"  Voxel size: {cltimg.get_voxel_size(affine_mni)}")
print("-------------------------------")
print(" ")

print("Test 2: Voxel size of an anisotropic DWI acquisition...")
print(f"  Voxel size: {cltimg.get_voxel_size(affine_dwi)}")
print("-------------------------------")
print(" ")

print("Test 3: Voxel size of an oblique acquisition (rotation does not change the size)...")
vx, vy, vz = cltimg.get_voxel_size(affine_obl)
print(f"  Voxel size: {vx:.2f} x {vy:.2f} x {vz:.2f} mm")
print("-------------------------------")


### 2.2 `get_voxel_volume`
<a id="image-get_voxel_volume"></a>

Compute the voxel volume from an affine matrix.

In [ ]:
########################### Testing get_voxel_volume ###########################
import clabtoolkit.imagetools as cltimg
import numpy as np

# MNI152 1 mm affine (radiological flip on X, origin at the anterior commissure)
affine_mni = np.array([[-1.0, 0.0, 0.0,   90.0],
                       [ 0.0, 1.0, 0.0, -126.0],
                       [ 0.0, 0.0, 1.0,  -72.0],
                       [ 0.0, 0.0, 0.0,    1.0]])

# Typical DWI affine with anisotropic voxels (2 x 2 x 2.5 mm)
affine_dwi = np.diag([2.0, 2.0, 2.5, 1.0])
affine_dwi[:3, 3] = [-96.0, -96.0, -60.0]

# Oblique acquisition: 15 degrees rotation around Z with 2 mm isotropic voxels
theta = np.deg2rad(15)
rot_z = np.array([[np.cos(theta), -np.sin(theta), 0.0],
                  [np.sin(theta),  np.cos(theta), 0.0],
                  [0.0,            0.0,           1.0]])
affine_obl = np.eye(4)
affine_obl[:3, :3] = rot_z @ np.diag([2.0, 2.0, 2.0])
affine_obl[:3, 3] = [-80.0, -100.0, -50.0]

print("Test 1: Voxel volume of the different affines...")
for name, aff in [("MNI 1 mm", affine_mni), ("DWI 2x2x2.5 mm", affine_dwi), ("Oblique 2 mm", affine_obl)]:
    print(f"  {name:16s}: {cltimg.get_voxel_volume(aff):.3f} mm3")
print("-------------------------------")
print(" ")

print("Test 2: Physical volume of a region (number of voxels x voxel volume)...")
mask = np.zeros((96, 96, 48), dtype=bool)
mask[40:50, 40:50, 20:30] = True           # 1000 voxels
vol_mm3 = mask.sum() * cltimg.get_voxel_volume(affine_dwi)
print(f"  {mask.sum()} voxels -> {vol_mm3:.1f} mm3 ({vol_mm3 / 1000:.1f} ml)")
print("-------------------------------")


### 2.3 `get_center`
<a id="image-get_center"></a>

Compute the center of the image from a NIfTI affine matrix.

In [ ]:
########################### Testing get_center ###########################
import clabtoolkit.imagetools as cltimg
import numpy as np

# MNI152 1 mm affine (radiological flip on X, origin at the anterior commissure)
affine_mni = np.array([[-1.0, 0.0, 0.0,   90.0],
                       [ 0.0, 1.0, 0.0, -126.0],
                       [ 0.0, 0.0, 1.0,  -72.0],
                       [ 0.0, 0.0, 0.0,    1.0]])

# Typical DWI affine with anisotropic voxels (2 x 2 x 2.5 mm)
affine_dwi = np.diag([2.0, 2.0, 2.5, 1.0])
affine_dwi[:3, 3] = [-96.0, -96.0, -60.0]

# Oblique acquisition: 15 degrees rotation around Z with 2 mm isotropic voxels
theta = np.deg2rad(15)
rot_z = np.array([[np.cos(theta), -np.sin(theta), 0.0],
                  [np.sin(theta),  np.cos(theta), 0.0],
                  [0.0,            0.0,           1.0]])
affine_obl = np.eye(4)
affine_obl[:3, :3] = rot_z @ np.diag([2.0, 2.0, 2.0])
affine_obl[:3, 3] = [-80.0, -100.0, -50.0]

print("Test 1: World-space origin of the MNI152 affine (mm location of voxel (0, 0, 0))...")
print(f"  Origin: {cltimg.get_center(affine_mni)}")
print("-------------------------------")
print(" ")

print("Test 2: The origin matches the mm coordinates of voxel (0, 0, 0) computed with vox2mm...")
print(f"  get_center: {cltimg.get_center(affine_dwi)}")
print(f"  vox2mm    : {cltimg.vox2mm(np.array([[0, 0, 0]]), affine_dwi)[0]}")
print("-------------------------------")
print(" ")

print("Test 3: The origin is not the geometric center of the volume. Computing the latter with vox2mm...")
shape = (182, 218, 182)
center_vox = (np.array(shape) - 1) / 2
print(f"  Origin          : {cltimg.get_center(affine_mni)}")
print(f"  Geometric center: {cltimg.vox2mm(center_vox[None, :], affine_mni)[0]}")
print("-------------------------------")


### 2.4 `get_rotation_matrix`
<a id="image-get_rotation_matrix"></a>

Extract the normalized rotation matrix from an affine matrix.

In [ ]:
########################### Testing get_rotation_matrix ###########################
import clabtoolkit.imagetools as cltimg
import numpy as np

# MNI152 1 mm affine (radiological flip on X, origin at the anterior commissure)
affine_mni = np.array([[-1.0, 0.0, 0.0,   90.0],
                       [ 0.0, 1.0, 0.0, -126.0],
                       [ 0.0, 0.0, 1.0,  -72.0],
                       [ 0.0, 0.0, 0.0,    1.0]])

# Typical DWI affine with anisotropic voxels (2 x 2 x 2.5 mm)
affine_dwi = np.diag([2.0, 2.0, 2.5, 1.0])
affine_dwi[:3, 3] = [-96.0, -96.0, -60.0]

# Oblique acquisition: 15 degrees rotation around Z with 2 mm isotropic voxels
theta = np.deg2rad(15)
rot_z = np.array([[np.cos(theta), -np.sin(theta), 0.0],
                  [np.sin(theta),  np.cos(theta), 0.0],
                  [0.0,            0.0,           1.0]])
affine_obl = np.eye(4)
affine_obl[:3, :3] = rot_z @ np.diag([2.0, 2.0, 2.0])
affine_obl[:3, 3] = [-80.0, -100.0, -50.0]

print("Test 1: Rotation matrix of the MNI152 affine (only the X flip remains)...")
print(cltimg.get_rotation_matrix(affine_mni))
print("-------------------------------")
print(" ")

print("Test 2: Rotation matrix of an anisotropic affine (the scaling is removed)...")
print(cltimg.get_rotation_matrix(affine_dwi))
print("-------------------------------")
print(" ")

print("Test 3: Recovering the rotation of the oblique acquisition...")
R = cltimg.get_rotation_matrix(affine_obl)
print(np.round(R, 4))
print(f"  Orthonormal (R @ R.T == I): {np.allclose(R @ R.T, np.eye(3))}")
print(f"  Recovered angle around Z: {np.rad2deg(np.arctan2(R[1, 0], R[0, 0])):.1f} degrees")
print("-------------------------------")


### 2.5 `get_vox_neighbors`
<a id="image-get_vox_neighbors"></a>

Get the neighborhood coordinates for a voxel.

In [ ]:
########################### Testing get_vox_neighbors ###########################
import clabtoolkit.imagetools as cltimg
import numpy as np

center = np.array([10, 15, 20])

print("Test 1: Number of neighbors of a 3D voxel for each neighborhood type...")
for nb in ["6", "12", "18", "26"]:
    neigh = cltimg.get_vox_neighbors(center, neighborhood=nb, dims="3")
    print(f"  {nb:>2s}-neighborhood: {len(neigh)} neighbors")
print(f"  6-neighbors of {center.tolist()}:\n{cltimg.get_vox_neighbors(center, neighborhood='6')}")
print("-------------------------------")
print(" ")

print("Test 2: Neighbors of a 2D pixel (4 and 8 connectivity)...")
pix = np.array([5, 5])
print(f"  4-neighbors: {cltimg.get_vox_neighbors(pix, neighborhood='4', dims='2').tolist()}")
print(f"  8-neighbors: {len(cltimg.get_vox_neighbors(pix, neighborhood='8', dims='2'))} pixels")
print("-------------------------------")
print(" ")

print("Test 3: Discarding the neighbors outside the volume for a voxel on the corner...")
shape = (20, 20, 20)
neigh = cltimg.get_vox_neighbors(np.array([0, 0, 0]), neighborhood="26")
inside = neigh[np.all((neigh >= 0) & (neigh < shape), axis=1)]
print(f"  Neighbors: {len(neigh)}, inside the volume: {len(inside)}")
print("-------------------------------")
print(" ")

print("Test 4: Mismatch between the coordinates and the number of dimensions...")
try:
    cltimg.get_vox_neighbors(np.array([5, 5]), neighborhood="26", dims="3")
except ValueError as e:
    print(f"  ValueError: {e}")
print("-------------------------------")


---
## <a id="image-operations"></a>3. Operations over Images
Methods to operate over images (e.g. crop).

### 3.1 `crop_image_from_mask`
<a id="image-crop_image_from_mask"></a>

Crop an image using a mask to the minimum bounding box containing the specified structures.

In [ ]:
########################### Testing crop_image_from_mask ###########################
import clabtoolkit.imagetools as cltimg
import clabtoolkit.parcellationtools as cltparc
import nibabel as nib
import numpy as np
import os

out_dir = "/tmp/imagetools_examples"
os.makedirs(out_dir, exist_ok=True)

# Simulated parcellation of 10 regions restricted to a spherical "brain" (zeros outside)
parc = cltparc.Parcellation.simulate_parcellation(n_regions=10, dimensions=(64, 64, 48), seed=42)
zz, yy, xx = np.ogrid[:64, :64, :48]
brain = (zz - 32) ** 2 + (yy - 32) ** 2 + (xx - 24) ** 2 <= 20**2
parc_data = (parc.data * brain).astype(np.int16)

parc_file = os.path.join(out_dir, "sub-01_atlas-sim_dseg.nii.gz")
mask_file = os.path.join(out_dir, "sub-01_desc-brain_mask.nii.gz")
nib.save(nib.Nifti1Image(parc_data, parc.affine), parc_file)
nib.save(nib.Nifti1Image(brain.astype(np.uint8), parc.affine), mask_file)

# Simulated T1-weighted image (3D) and BOLD-like image (4D, 5 volumes) over the whole field of view
t1_file = os.path.join(out_dir, "sub-01_T1w.nii.gz")
bold_file = os.path.join(out_dir, "sub-01_bold.nii.gz")
cltimg.simulate_image((64, 64, 48), t1_file, affine=parc.affine, distribution="uniform", low=100, high=1000, random_seed=0)
cltimg.simulate_image((64, 64, 48), bold_file, affine=parc.affine, n_volumes=5, random_seed=0)
print(" ")

print("Test 1: Cropping the T1w image to the bounding box of the binary brain mask (file path)...")
crop_file = cltimg.crop_image_from_mask(t1_file, mask_file, os.path.join(out_dir, "sub-01_desc-crop_T1w.nii.gz"))
crop = nib.load(crop_file)
print(f"  Original shape: {nib.load(t1_file).shape}, cropped shape: {crop.shape}")
print(f"  The affine is updated so the cropped voxels keep their mm position. New origin: {crop.affine[:3, 3]}")
print("-------------------------------")
print(" ")

print("Test 2: Cropping to specific structures (regions 3 and 7) using the parcellation as a numpy array...")
crop_st = cltimg.crop_image_from_mask(t1_file, parc_data, os.path.join(out_dir, "sub-01_desc-crop37_T1w.nii.gz"), st_codes=[3, 7])
crop_st_img = nib.load(crop_st)
print(f"  Cropped shape: {crop_st_img.shape}, non-zero voxels: {np.count_nonzero(crop_st_img.get_fdata())}")
print("-------------------------------")
print(" ")

print("Test 3: Cropping a 4D image. The mask is applied to every volume...")
crop4d = cltimg.crop_image_from_mask(bold_file, mask_file, os.path.join(out_dir, "sub-01_desc-crop_bold.nii.gz"))
print(f"  Original shape: {nib.load(bold_file).shape}, cropped shape: {nib.load(crop4d).shape}")
print("-------------------------------")
print(" ")

print("Test 4: Using a mask with a different shape than the image...")
try:
    cltimg.crop_image_from_mask(t1_file, np.ones((10, 10, 10)), os.path.join(out_dir, "bad.nii.gz"))
except ValueError as e:
    print(f"  ValueError: {e}")
print("-------------------------------")


### 3.2 `cropped_to_native`
<a id="image-cropped_to_native"></a>

Restore a cropped image to the dimensions of a reference native image.

In [ ]:
########################### Testing cropped_to_native ###########################
import clabtoolkit.imagetools as cltimg
import clabtoolkit.parcellationtools as cltparc
import nibabel as nib
import numpy as np
import os

out_dir = "/tmp/imagetools_examples"
os.makedirs(out_dir, exist_ok=True)

# Simulated parcellation of 10 regions restricted to a spherical "brain" (zeros outside)
parc = cltparc.Parcellation.simulate_parcellation(n_regions=10, dimensions=(64, 64, 48), seed=42)
zz, yy, xx = np.ogrid[:64, :64, :48]
brain = (zz - 32) ** 2 + (yy - 32) ** 2 + (xx - 24) ** 2 <= 20**2
parc_data = (parc.data * brain).astype(np.int16)

parc_file = os.path.join(out_dir, "sub-01_atlas-sim_dseg.nii.gz")
mask_file = os.path.join(out_dir, "sub-01_desc-brain_mask.nii.gz")
nib.save(nib.Nifti1Image(parc_data, parc.affine), parc_file)
nib.save(nib.Nifti1Image(brain.astype(np.uint8), parc.affine), mask_file)

# Simulated T1-weighted image (3D) and BOLD-like image (4D, 5 volumes) over the whole field of view
t1_file = os.path.join(out_dir, "sub-01_T1w.nii.gz")
bold_file = os.path.join(out_dir, "sub-01_bold.nii.gz")
cltimg.simulate_image((64, 64, 48), t1_file, affine=parc.affine, distribution="uniform", low=100, high=1000, random_seed=0)
cltimg.simulate_image((64, 64, 48), bold_file, affine=parc.affine, n_volumes=5, random_seed=0)
print(" ")

crop_file = cltimg.crop_image_from_mask(t1_file, mask_file, os.path.join(out_dir, "sub-01_desc-crop_T1w.nii.gz"))

print("Test 1: Restoring the cropped T1w image to the native field of view...")
native_file = cltimg.cropped_to_native(crop_file, t1_file, os.path.join(out_dir, "sub-01_desc-restored_T1w.nii.gz"))
restored = nib.load(native_file).get_fdata()
cropped = nib.load(crop_file).get_fdata()
print(f"  Cropped shape: {cropped.shape} -> restored shape: {restored.shape}")
print(f"  Non-zero voxels: cropped={np.count_nonzero(cropped)}, restored={np.count_nonzero(restored)}")
print("-------------------------------")
print(" ")

print("Test 2: Each restored voxel is back in its original position...")
nz = restored != 0
orig = nib.load(t1_file).get_fdata()
print(f"  Restored values equal the original ones: {np.allclose(restored[nz], orig[nz])}")
print("-------------------------------")
print(" ")

print("Test 3: Restoring a cropped 4D image...")
crop4d = cltimg.crop_image_from_mask(bold_file, mask_file, os.path.join(out_dir, "sub-01_desc-crop_bold.nii.gz"))
native4d = cltimg.cropped_to_native(crop4d, bold_file, os.path.join(out_dir, "sub-01_desc-restored_bold.nii.gz"))
print(f"  Cropped shape: {nib.load(crop4d).shape} -> restored shape: {nib.load(native4d).shape}")
print("-------------------------------")


---
## <a id="image-transformations"></a>4. Transformations and Changes of Space
Methods to apply transformations or changes of space.

### 4.1 `apply_multi_transf`
<a id="image-apply_multi_transf"></a>

Apply an ANTs transformation to an image with support for multiple transform types.

In [ ]:
########################### Testing apply_multi_transf ###########################
import clabtoolkit.imagetools as cltimg
import clabtoolkit.parcellationtools as cltparc
import nibabel as nib
import numpy as np
import scipy.io
import shutil
import os

# NOTE: This example requires ANTs (antsApplyTransforms) in the PATH.
#       Alternatively use cont_tech="singularity" or "docker" together with cont_image.
if shutil.which("antsApplyTransforms") is None:
    print("antsApplyTransforms was not found in the PATH. Skipping the example.")
else:
    xfm_dir = "/tmp/imagetools_examples/xfm"
    os.makedirs(xfm_dir, exist_ok=True)

    # Simulated parcellation restricted to a central sphere, used as moving and reference image
    parc = cltparc.Parcellation.simulate_parcellation(n_regions=10, dimensions=(48, 48, 40), seed=7)
    zz, yy, xx = np.ogrid[:48, :48, :40]
    brain = (zz - 24) ** 2 + (yy - 24) ** 2 + (xx - 20) ** 2 <= 14**2
    in_file = "/tmp/imagetools_examples/sub-01_atlas-sim_dseg_48.nii.gz"
    nib.save(nib.Nifti1Image((parc.data * brain).astype(np.int16), parc.affine), in_file)

    # ANTs affine transform (ITK .mat format): a pure translation of 6 mm along the first axis.
    # ANTs works in LPS coordinates, so the content moves +6 mm along X in the RAS space of the NIfTI file.
    # apply_multi_transf looks for "<xfm_output>_desc-affine_xfm.mat" (and the optional warps).
    xfm_output = os.path.join(xfm_dir, "sub-01_from-T1w_to-shifted")
    params = np.array([1, 0, 0, 0, 1, 0, 0, 0, 1, 6.0, 0, 0], dtype=np.float64)[:, None]
    scipy.io.savemat(xfm_output + "_desc-affine_xfm.mat",
                     {"AffineTransform_double_3_3": params, "fixed": np.zeros((3, 1))}, format="4")

    def centroid_mm(file):
        """Centroid (mm) of all the labeled voxels."""
        img = nib.load(file)
        data = np.squeeze(img.get_fdata())          # antsApplyTransforms -e 3 writes a 4D image with one volume
        vox = np.argwhere(data > 0).mean(axis=0)
        return np.round(cltimg.vox2mm(vox[None, :], img.affine)[0], 2)

    print("Test 1: Applying the affine transform with nearest neighbor interpolation (labels are preserved)...")
    out_file = "/tmp/imagetools_examples/sub-01_space-shifted_dseg.nii.gz"
    cltimg.apply_multi_transf(in_file, out_file, in_file, xfm_output, interp_order=0, overwrite=True)
    print(f"  Labels in the output: {np.unique(nib.load(out_file).get_fdata()).astype(int)}")
    print(f"  Centroid (mm): before {centroid_mm(in_file)}, after {centroid_mm(out_file)}")
    print("-------------------------------")
    print(" ")

    print("Test 2: Applying the inverse transform brings the image back to the original position...")
    back_file = "/tmp/imagetools_examples/sub-01_space-back_dseg.nii.gz"
    cltimg.apply_multi_transf(out_file, back_file, in_file, xfm_output, interp_order=0, invert=True, overwrite=True)
    print(f"  Centroid (mm) after the inverse: {centroid_mm(back_file)}")
    print(f"  Identical to the input: {np.array_equal(np.squeeze(nib.load(back_file).get_fdata()), nib.load(in_file).get_fdata())}")
    print("-------------------------------")
    print(" ")

    print("Test 3: Errors for an unsupported interpolation and a missing transform...")
    try:
        cltimg.apply_multi_transf(in_file, out_file, in_file, xfm_output, interp_order=9)
    except ValueError as e:
        print(f"  ValueError: {e}")
    try:
        cltimg.apply_multi_transf(in_file, "/tmp/imagetools_examples/missing.nii.gz", in_file,
                                  os.path.join(xfm_dir, "sub-01_from-T1w_to-missing"), overwrite=True)
    except FileNotFoundError as e:
        print(f"  FileNotFoundError: {e}")
    print("-------------------------------")


### 4.2 `vox2mm`
<a id="image-vox2mm"></a>

Convert voxel coordinates to millimeter coordinates using an affine matrix.

In [ ]:
########################### Testing vox2mm ###########################
import clabtoolkit.imagetools as cltimg
import numpy as np

# MNI152 1 mm affine (radiological flip on X, origin at the anterior commissure)
affine_mni = np.array([[-1.0, 0.0, 0.0,   90.0],
                       [ 0.0, 1.0, 0.0, -126.0],
                       [ 0.0, 0.0, 1.0,  -72.0],
                       [ 0.0, 0.0, 0.0,    1.0]])

# Typical DWI affine with anisotropic voxels (2 x 2 x 2.5 mm)
affine_dwi = np.diag([2.0, 2.0, 2.5, 1.0])
affine_dwi[:3, 3] = [-96.0, -96.0, -60.0]

# Oblique acquisition: 15 degrees rotation around Z with 2 mm isotropic voxels
theta = np.deg2rad(15)
rot_z = np.array([[np.cos(theta), -np.sin(theta), 0.0],
                  [np.sin(theta),  np.cos(theta), 0.0],
                  [0.0,            0.0,           1.0]])
affine_obl = np.eye(4)
affine_obl[:3, :3] = rot_z @ np.diag([2.0, 2.0, 2.0])
affine_obl[:3, 3] = [-80.0, -100.0, -50.0]

print("Test 1: Converting the corner and center voxels of the MNI152 template to mm...")
vox = np.array([[0, 0, 0], [181, 217, 181], [90, 126, 72]])
print(cltimg.vox2mm(vox, affine_mni))
print("-------------------------------")
print(" ")

print("Test 2: The coordinates can also be given as a 3 x N matrix (it is transposed automatically)...")
vox_3xN = np.array([[0, 10, 20, 30], [0, 10, 20, 30], [0, 10, 20, 30]])
print(cltimg.vox2mm(vox_3xN, affine_dwi))
print("-------------------------------")
print(" ")

print("Test 3: Converting fractional voxel coordinates (e.g. a centroid) in an oblique image...")
print(np.round(cltimg.vox2mm(np.array([[12.5, 30.25, 18.0]]), affine_obl), 3))
print("-------------------------------")
print(" ")

print("Test 4: Coordinates with a wrong number of columns...")
try:
    cltimg.vox2mm(np.array([[1, 2], [3, 4]]), affine_mni)
except ValueError as e:
    print(f"  ValueError: {e}")
print("-------------------------------")


### 4.3 `mm2vox`
<a id="image-mm2vox"></a>

Convert millimeter coordinates to voxel coordinates using an affine matrix.

In [ ]:
########################### Testing mm2vox ###########################
import clabtoolkit.imagetools as cltimg
import numpy as np

# MNI152 1 mm affine (radiological flip on X, origin at the anterior commissure)
affine_mni = np.array([[-1.0, 0.0, 0.0,   90.0],
                       [ 0.0, 1.0, 0.0, -126.0],
                       [ 0.0, 0.0, 1.0,  -72.0],
                       [ 0.0, 0.0, 0.0,    1.0]])

# Typical DWI affine with anisotropic voxels (2 x 2 x 2.5 mm)
affine_dwi = np.diag([2.0, 2.0, 2.5, 1.0])
affine_dwi[:3, 3] = [-96.0, -96.0, -60.0]

# Oblique acquisition: 15 degrees rotation around Z with 2 mm isotropic voxels
theta = np.deg2rad(15)
rot_z = np.array([[np.cos(theta), -np.sin(theta), 0.0],
                  [np.sin(theta),  np.cos(theta), 0.0],
                  [0.0,            0.0,           1.0]])
affine_obl = np.eye(4)
affine_obl[:3, :3] = rot_z @ np.diag([2.0, 2.0, 2.0])
affine_obl[:3, 3] = [-80.0, -100.0, -50.0]

print("Test 1: The MNI origin (0, 0, 0) mm corresponds to voxel (90, 126, 72)...")
print(cltimg.mm2vox(np.array([[0.0, 0.0, 0.0]]), affine_mni))
print("-------------------------------")
print(" ")

print("Test 2: Round trip voxel -> mm -> voxel in an oblique image...")
vox = np.array([[3, 7, 11], [40, 2, 25]])
back = cltimg.mm2vox(cltimg.vox2mm(vox, affine_obl), affine_obl)
print(f"  Recovered: {np.round(back, 6).tolist()}, identical: {np.allclose(back, vox)}")
print("-------------------------------")
print(" ")

print("Test 3: Finding the voxel indices of points given in mm (e.g. coordinates reported in a paper)...")
peaks_mm = np.array([[-42.0, -22.0, 54.0],   # Left motor cortex
                     [ 40.0, -60.0, 44.0]])  # Right parietal cortex
vox = cltimg.mm2vox(peaks_mm, affine_mni)
print(f"  Voxel coordinates: {vox.tolist()}")
print(f"  Indices to read the image: {np.round(vox).astype(int).tolist()}")
print("-------------------------------")


---
## <a id="image-4d"></a>5. 4D Images
Methods to work with 4D images.

### 5.1 `merge_to_4d`
<a id="image-merge_to_4d"></a>

Merge multiple 3D or 4D NIfTI files into a single 4D NIfTI file.

In [ ]:
########################### Testing merge_to_4d ###########################
import clabtoolkit.imagetools as cltimg
import nibabel as nib
import numpy as np
import json
import os

out_dir = "/tmp/imagetools_examples/merge"
os.makedirs(out_dir, exist_ok=True)
affine = np.diag([2.0, 2.0, 2.0, 1.0])

# Three 3D runs and one 4D run with the same geometry
runs_3d = []
for i in range(3):
    f = os.path.join(out_dir, f"sub-01_run-{i + 1}_T2w.nii.gz")
    cltimg.simulate_image((32, 32, 20), f, affine=affine, random_seed=i)
    runs_3d.append(f)
run_4d = os.path.join(out_dir, "sub-01_run-4_T2w.nii.gz")
cltimg.simulate_image((32, 32, 20), run_4d, affine=affine, n_volumes=4, random_seed=3)
print(" ")

print("Test 1: Merging three 3D images in memory (no output file, no sidecars)...")
merged, meta = cltimg.merge_to_4d(runs_3d)
print(f"  Merged shape: {merged.shape}, metadata: {meta}")
print("-------------------------------")
print(" ")

print("Test 2: Merging a mix of 3D and 4D images and saving the result...")
out_file = os.path.join(out_dir, "sub-01_desc-merged_T2w.nii.gz")
merged, meta = cltimg.merge_to_4d(runs_3d + [run_4d], output_path=out_file)
print(f"  Saved shape: {nib.load(out_file).shape}")
print("-------------------------------")
print(" ")

print("Test 3: Merging two DWI runs. The bvec, bval and JSON sidecars are concatenated...")
dwi_files = []
for i, n_dirs in enumerate([6, 4]):
    stem = os.path.join(out_dir, f"sub-01_run-{i + 1}_dwi")
    cltimg.simulate_image((32, 32, 20), stem + ".nii.gz", affine=affine, n_volumes=n_dirs, random_seed=i)
    bvecs = np.random.default_rng(i).normal(size=(3, n_dirs))
    bvecs /= np.linalg.norm(bvecs, axis=0)
    bvecs[:, 0] = 0                                              # First volume is a B0
    np.savetxt(stem + ".bvec", bvecs, fmt="%.6f")
    np.savetxt(stem + ".bval", np.r_[0, np.full(n_dirs - 1, 1000 * (i + 1))][None, :], fmt="%d")
    with open(stem + ".json", "w") as f:
        json.dump({"RepetitionTime": 3.2, "PhaseEncodingDirection": "j-" if i == 0 else "j"}, f)
    dwi_files.append(stem + ".nii.gz")
print(" ")
merged_dwi, meta_dwi = cltimg.merge_to_4d(dwi_files, output_path=os.path.join(out_dir, "sub-01_dwi.nii.gz"))
print(f"  bvecs: {meta_dwi['bvecs'].shape}, bvals: {meta_dwi['bvals'].astype(int)}")
print(f"  Merged JSON: {meta_dwi['json']}")
print(f"  Files written: {sorted(f for f in os.listdir(out_dir) if f.startswith('sub-01_dwi'))}")
print("-------------------------------")
print(" ")

print("Test 4: Merging images with different affines...")
other = os.path.join(out_dir, "sub-01_run-5_T2w.nii.gz")
cltimg.simulate_image((32, 32, 20), other, affine=np.diag([1.0, 1.0, 1.0, 1.0]), random_seed=5)
try:
    cltimg.merge_to_4d([runs_3d[0], other])
except (ValueError, OSError) as e:
    print(f"  {type(e).__name__}: {e}")
print("-------------------------------")


### 5.2 `create_spams`
<a id="image-create_spams"></a>

Create SPAM probability maps from multiple parcellation images.

In [ ]:
########################### Testing create_spams ###########################
import clabtoolkit.imagetools as cltimg
import clabtoolkit.parcellationtools as cltparc
import nibabel as nib
import numpy as np
import os

out_dir = "/tmp/imagetools_examples/spams"
os.makedirs(out_dir, exist_ok=True)

# Five "subjects": the same 6-region parcellation shifted by a few voxels to mimic the inter-subject variability
parc = cltparc.Parcellation.simulate_parcellation(n_regions=6, dimensions=(40, 40, 30), seed=100)
rng = np.random.default_rng(0)
parc_files = []
for subj in range(5):
    shifted = np.roll(parc.data, shift=tuple(rng.integers(-2, 3, size=3)), axis=(0, 1, 2))
    f = os.path.join(out_dir, f"sub-{subj + 1:02d}_atlas-sim_dseg.nii.gz")
    nib.save(nib.Nifti1Image(shifted.astype(np.int16), parc.affine), f)
    parc_files.append(f)

# FreeSurfer-style lookup table for the 6 regions (index name R G B A)
lut_file = os.path.join(out_dir, "atlas-sim.lut")
colors = [(230, 25, 75), (60, 180, 75), (255, 225, 25), (0, 130, 200), (245, 130, 48), (145, 30, 180)]
with open(lut_file, "w") as f:
    for i, (r, g, b) in enumerate(colors, start=1):
        f.write(f"{i}  region-{i:02d}  {r} {g} {b} 0\n")

print("Test 1: Creating the SPAMs using a lookup table file...")
spam_file = os.path.join(out_dir, "atlas-sim_desc-spam.nii.gz")
cltimg.create_spams(parc_files, spam_file, lut_table=lut_file)
spam = nib.load(spam_file).get_fdata()
print(f"  SPAM shape: {spam.shape} (one probability map per region)")
print(f"  Probabilities sum to 1 at every voxel: {np.allclose(spam.sum(axis=3), 1)}")
print(f"  Voxels with full agreement across subjects (p = 1): {int((spam.max(axis=3) == 1).sum())} of {int(np.prod(spam.shape[:3]))}")
print("-------------------------------")
print(" ")

print("Test 2: Saving also an RGB color-coded SPAM image (colors weighted by the probabilities)...")
cltimg.create_spams(parc_files, spam_file, lut_table=lut_file, save_color_spams=True)
print(f"  Files: {sorted(f for f in os.listdir(out_dir) if 'spam' in f)}")
print("-------------------------------")
print(" ")

print("Test 3: Passing the lookup table as a dictionary (index, name and color keys)...")
lut_dict = {
    "index": list(range(1, 7)),
    "name": [f"region-{i:02d}" for i in range(1, 7)],
    "color": ["#e6194b", "#3cb44b", "#ffe119", "#0082c8", "#f58230", "#911eb4"],
}
spam_file_dict = os.path.join(out_dir, "atlas-sim_desc-spamdict.nii.gz")
cltimg.create_spams(parc_files, spam_file_dict, lut_table=lut_dict, save_color_spams=True)
print(f"  SPAM shape: {nib.load(spam_file_dict).shape}, colored: {nib.load(spam_file_dict.replace('.nii.gz', '_colored.nii.gz')).shape}")
print("-------------------------------")
print(" ")

print("Test 4: Creating the SPAMs without a lookup table (the regions are taken from the images)...")
spam_file_nolut = os.path.join(out_dir, "atlas-sim_desc-spamnolut.nii.gz")
cltimg.create_spams(parc_files, spam_file_nolut, lut_table=None)
print(f"  SPAM shape: {nib.load(spam_file_nolut).shape}")
print("-------------------------------")


### 5.3 `spams2maxprob`
<a id="image-spams2maxprob"></a>

Convert SPAM probability maps to a maximum probability parcellation.

In [ ]:
########################### Testing spams2maxprob ###########################
import clabtoolkit.imagetools as cltimg
import clabtoolkit.parcellationtools as cltparc
import nibabel as nib
import numpy as np
import os

out_dir = "/tmp/imagetools_examples/spams"
os.makedirs(out_dir, exist_ok=True)

# Five "subjects": the same 6-region parcellation shifted by a few voxels to mimic the inter-subject variability
parc = cltparc.Parcellation.simulate_parcellation(n_regions=6, dimensions=(40, 40, 30), seed=100)
rng = np.random.default_rng(0)
parc_files = []
for subj in range(5):
    shifted = np.roll(parc.data, shift=tuple(rng.integers(-2, 3, size=3)), axis=(0, 1, 2))
    f = os.path.join(out_dir, f"sub-{subj + 1:02d}_atlas-sim_dseg.nii.gz")
    nib.save(nib.Nifti1Image(shifted.astype(np.int16), parc.affine), f)
    parc_files.append(f)

# FreeSurfer-style lookup table for the 6 regions (index name R G B A)
lut_file = os.path.join(out_dir, "atlas-sim.lut")
colors = [(230, 25, 75), (60, 180, 75), (255, 225, 25), (0, 130, 200), (245, 130, 48), (145, 30, 180)]
with open(lut_file, "w") as f:
    for i, (r, g, b) in enumerate(colors, start=1):
        f.write(f"{i}  region-{i:02d}  {r} {g} {b} 0\n")
spam_file = os.path.join(out_dir, "atlas-sim_desc-spam.nii.gz")
cltimg.create_spams(parc_files, spam_file, lut_table=lut_file)

print("Test 1: Converting the SPAMs to a maximum probability parcellation (returned as an array)...")
maxprob = cltimg.spams2maxprob(spam_file)
print(f"  Shape: {maxprob.shape}, labels: {np.unique(maxprob)}")
print("-------------------------------")
print(" ")

print("Test 2: Applying a probability threshold of 0.7 and saving the result...")
maxp_file = os.path.join(out_dir, "atlas-sim_desc-maxprob70_dseg.nii.gz")
cltimg.spams2maxprob(spam_file, prob_thresh=0.7, maxp_name=maxp_file)
maxp70 = nib.load(maxp_file).get_fdata()
print(f"  Unlabeled voxels: thresh 0.05 -> {(maxprob == 0).sum()}, thresh 0.7 -> {int((maxp70 == 0).sum())}")
print("-------------------------------")
print(" ")

print("Test 3: Keeping only the first three regions (volumes 0, 1 and 2)...")
maxp_sub = cltimg.spams2maxprob(spam_file, vol_indexes=[0, 1, 2])
print(f"  Labels: {np.unique(maxp_sub)}")
print("-------------------------------")


### 5.4 `simulate_image`
<a id="image-simulate_image"></a>

Generate a simulated image with random values at non-zero voxel positions.

In [ ]:
########################### Testing simulate_image ###########################
import clabtoolkit.imagetools as cltimg
import clabtoolkit.parcellationtools as cltparc
import nibabel as nib
import numpy as np
import os

out_dir = "/tmp/imagetools_examples"
os.makedirs(out_dir, exist_ok=True)

print("Test 1: Simulating a 3D image from a shape (every voxel gets a value from a normal distribution)...")
img = cltimg.simulate_image((64, 64, 40), os.path.join(out_dir, "sim_normal.nii.gz"), loc=100, scale=15, random_seed=42)
data = img.get_fdata()
print(f"  Mean: {data.mean():.2f}, std: {data.std():.2f}")
print("-------------------------------")
print(" ")

print("Test 2: Simulating a 4D image (10 volumes) with uniform values inside a Parcellation...")
parc = cltparc.Parcellation.simulate_parcellation(n_regions=8, dimensions=(48, 48, 32), seed=1)
img4d = cltimg.simulate_image(parc, os.path.join(out_dir, "sim_uniform_4d.nii.gz"), n_volumes=10,
                              distribution="uniform", low=0, high=100, random_seed=1)
print(f"  Range: [{img4d.get_fdata().min():.2f}, {img4d.get_fdata().max():.2f}], affine preserved: {np.allclose(img4d.affine, parc.affine)}")
print("-------------------------------")
print(" ")

print("Test 3: Simulating exponential values only inside a brain mask read from a file...")
mask = np.zeros((48, 48, 32), dtype=np.uint8)
mask[10:38, 10:38, 6:26] = 1
mask_file = os.path.join(out_dir, "sim_mask.nii.gz")
nib.save(nib.Nifti1Image(mask, np.eye(4)), mask_file)
img_exp = cltimg.simulate_image(mask_file, os.path.join(out_dir, "sim_exponential.nii.gz"),
                                distribution="exponential", scale=2.0, random_seed=3)
d = img_exp.get_fdata()
print(f"  Values outside the mask are zero: {np.all(d[mask == 0] == 0)}, mean inside: {d[mask == 1].mean():.2f}")
print("-------------------------------")
print(" ")

print("Test 4: Requesting an unsupported distribution...")
try:
    cltimg.simulate_image((10, 10, 10), os.path.join(out_dir, "bad.nii.gz"), distribution="poisson")
except ValueError as e:
    print(f"  ValueError: {e}")
print("-------------------------------")


### 5.5 `delete_volumes_from_4D_images`
<a id="image-delete_volumes_from_4D_images"></a>

Remove specific volumes from a 4D neuroimaging file.

In [ ]:
########################### Testing delete_volumes_from_4D_images ###########################
import clabtoolkit.imagetools as cltimg
import nibabel as nib
import numpy as np
import os

out_dir = "/tmp/imagetools_examples"
os.makedirs(out_dir, exist_ok=True)

# 4D image with 20 volumes. Each volume is filled with its own index to track which ones remain
data = np.broadcast_to(np.arange(20, dtype=np.float32), (16, 16, 10, 20)).copy()
in_file = os.path.join(out_dir, "sub-01_task-rest_bold.nii.gz")
nib.save(nib.Nifti1Image(data, np.eye(4)), in_file)

def kept(file):
    return nib.load(file).get_fdata()[0, 0, 0, :].astype(int).tolist()

print("Test 1: Removing the first three dummy scans and the last volume...")
out_file, removed = cltimg.delete_volumes_from_4D_images(in_file, os.path.join(out_dir, "sub-01_desc-clean_bold.nii.gz"),
                                                         vols_to_delete=[0, 1, 2, 19], overwrite=True)
print(f"  Volumes kept: {kept(out_file)}")
print("-------------------------------")
print(" ")

print("Test 2: Using ranges (tuples) and the string syntax ('5-7, 10:2:14')...")
out_file, removed = cltimg.delete_volumes_from_4D_images(in_file, os.path.join(out_dir, "sub-01_desc-subset_bold.nii.gz"),
                                                         vols_to_delete=[(0, 1), "5-7, 10:2:14"], overwrite=True)
print(f"  Removed: {removed}")
print(f"  Volumes kept: {kept(out_file)}")
print("-------------------------------")
print(" ")

print("Test 3: Errors for an existing output without overwrite and for out-of-range volumes...")
try:
    cltimg.delete_volumes_from_4D_images(in_file, os.path.join(out_dir, "sub-01_desc-clean_bold.nii.gz"), vols_to_delete=[0])
except FileExistsError as e:
    print(f"  FileExistsError: {e}")
try:
    cltimg.delete_volumes_from_4D_images(in_file, os.path.join(out_dir, "tmp_bold.nii.gz"), vols_to_delete=[5, 25], overwrite=True)
except ValueError as e:
    print(f"  ValueError: {e}")
print("-------------------------------")


---
## <a id="image-arrays"></a>6. Operations from 3D or 4D Arrays
Methods to perform operations from 3D or 4D arrays.

### 6.1 `extract_mesh_from_volume`
<a id="image-extract_mesh_from_volume"></a>

Extract a surface mesh from a 3D volume using the marching cubes algorithm.

In [ ]:
########################### Testing extract_mesh_from_volume ###########################
import clabtoolkit.imagetools as cltimg
import numpy as np

# Sphere of radius 8 voxels centered at voxel (12, 20, 28) in a 40 x 40 x 40 volume
zz, yy, xx = np.ogrid[:40, :40, :40]
sphere = ((zz - 12) ** 2 + (yy - 20) ** 2 + (xx - 28) ** 2 <= 8**2).astype(np.uint8)
import clabtoolkit.parcellationtools as cltparc

print("Test 1: Extracting the mesh of the sphere in voxel coordinates, with and without Gaussian smoothing...")
mesh = cltimg.extract_mesh_from_volume(sphere)
mesh_raw = cltimg.extract_mesh_from_volume(sphere, gaussian_smooth=False)
print(f"  Smoothed: {mesh.n_points} vertices, {mesh.n_cells} faces")
print(f"  Raw     : {mesh_raw.n_points} vertices, {mesh_raw.n_cells} faces")
print(f"  Bounds (voxels): {np.round(mesh.bounds, 1)}")
print("-------------------------------")
print(" ")

print("Test 2: Extracting the mesh in mm using an affine with 2 mm voxels and more smoothing...")
affine = np.diag([2.0, 2.0, 2.0, 1.0])
affine[:3, 3] = [-40, -40, -40]
mesh_mm = cltimg.extract_mesh_from_volume(sphere, affine=affine, smooth_iterations=30)
print(f"  Bounds (mm): {np.round(mesh_mm.bounds, 1)}")
print(f"  Surface area: {mesh_mm.area:.1f} mm2 (analytical: {4 * np.pi * 16**2:.1f} mm2)")
print("-------------------------------")
print(" ")

print("Test 3: Extracting the mesh of a single region of a parcellation and tagging its vertices with the label...")
parc = cltparc.Parcellation.simulate_parcellation(n_regions=10, dimensions=(48, 48, 40), seed=3)
region = (parc.data == 4).astype(np.uint8)
mesh_reg = cltimg.extract_mesh_from_volume(region, affine=parc.affine, vertex_value=4)
print(f"  Vertices: {mesh_reg.n_points}, vertex values: {np.unique(mesh_reg.point_data['default'])}")
print("-------------------------------")
print(" ")

print("Test 4: Passing a 2D array...")
try:
    cltimg.extract_mesh_from_volume(np.ones((10, 10)))
except ValueError as e:
    print(f"  ValueError: {e}")
print("-------------------------------")


### 6.2 `extract_centroid_from_volume`
<a id="image-extract_centroid_from_volume"></a>

Extract the centroid and the voxel count from a 3D binary volume.

In [ ]:
########################### Testing extract_centroid_from_volume ###########################
import clabtoolkit.imagetools as cltimg
import numpy as np

# Sphere of radius 8 voxels centered at voxel (12, 20, 28) in a 40 x 40 x 40 volume
zz, yy, xx = np.ogrid[:40, :40, :40]
sphere = ((zz - 12) ** 2 + (yy - 20) ** 2 + (xx - 28) ** 2 <= 8**2).astype(np.uint8)

print("Test 1: Centroid and voxel count of the sphere without smoothing or closing...")
centroid, nvox = cltimg.extract_centroid_from_volume(sphere, gaussian_smooth=False, closing_iterations=0)
print(f"  Centroid (voxels): {centroid}, voxels: {nvox}")
print("-------------------------------")
print(" ")

print("Test 2: Morphological closing fills an off-center cavity, so the centroid goes back to the sphere center...")
cavity = sphere.copy()
cavity[11:14, 19:22, 30:33] = 0
centroid_open, nvox_cav = cltimg.extract_centroid_from_volume(cavity, gaussian_smooth=False, closing_iterations=0)
centroid_c, _ = cltimg.extract_centroid_from_volume(cavity, gaussian_smooth=False, closing_iterations=2)
print(f"  Centroid without closing: {np.round(centroid_open, 3)}, with closing: {np.round(centroid_c, 3)}")
print(f"  Voxel count (always from the input volume): {nvox_cav}")
affine = np.diag([2.0, 2.0, 2.0, 1.0])
print(f"  Centroid in mm (2 mm voxels): {cltimg.vox2mm(centroid_c[None, :], affine)[0]}")
print("-------------------------------")
print(" ")

print("Test 3: Gaussian smoothing removes a thin spur before computing the centroid. The voxel count is always the original one...")
spur = sphere.copy()
spur[12, 20, 37:40] = 1
c_raw, n_raw = cltimg.extract_centroid_from_volume(spur, gaussian_smooth=False, closing_iterations=0)
c_smooth, n_smooth = cltimg.extract_centroid_from_volume(spur, gaussian_smooth=True, sigma=1.0)
print(f"  Without smoothing: centroid {np.round(c_raw, 3)}, voxels {n_raw}")
print(f"  With smoothing   : centroid {np.round(c_smooth, 3)}, voxels {n_smooth}")
print("-------------------------------")
print(" ")

print("Test 4: An empty volume returns a NaN centroid and zero voxels...")
print(f"  {cltimg.extract_centroid_from_volume(np.zeros((10, 10, 10)))}")
print("-------------------------------")


### 6.3 `create_spams_from_volume`
<a id="image-create_spams_from_volume"></a>

Create SPAMs (Spatial Probability Maps) from individual parcellation volumes.

In [ ]:
########################### Testing create_spams_from_volume ###########################
import clabtoolkit.imagetools as cltimg
import clabtoolkit.parcellationtools as cltparc
import numpy as np

# Five "subjects": the same 6-region parcellation shifted by a few voxels, stacked as (X, Y, Z, subjects)
parc = cltparc.Parcellation.simulate_parcellation(n_regions=6, dimensions=(40, 40, 30), seed=100)
rng = np.random.default_rng(0)
indiv_parc = np.stack(
    [np.roll(parc.data, shift=tuple(rng.integers(-2, 3, size=3)), axis=(0, 1, 2)) for _ in range(5)],
    axis=-1,
).astype(np.int32)
print(f"Individual parcellations: {indiv_parc.shape}")

print("Test 1: Creating the SPAMs for all the regions (IDs taken with np.unique)...")
sts_ids = np.unique(indiv_parc)
spams = cltimg.create_spams_from_volume(indiv_parc, sts_ids)
print(f"  SPAM shape: {spams.shape}, probabilities sum to 1: {np.allclose(spams.sum(axis=3), 1)}")
print(f"  Probabilities at voxel (20, 20, 15): {spams[20, 20, 15]}")
print("-------------------------------")
print(" ")

print("Test 2: Creating the SPAMs only for regions 2 and 5...")
spams_25 = cltimg.create_spams_from_volume(indiv_parc, [2, 5])
print(f"  SPAM shape: {spams_25.shape}")
print(f"  Voxels where region 2 appears in at least 3 of 5 subjects: {(spams_25[..., 0] >= 0.6).sum()}")
print("-------------------------------")
print(" ")

print("Test 3: Passing a 3D array instead of a 4D one...")
try:
    cltimg.create_spams_from_volume(indiv_parc[..., 0], [1, 2])
except ValueError as e:
    print(f"  ValueError: {e}")
print("-------------------------------")


### 6.4 `spams2maxprob_from_volume`
<a id="image-spams2maxprob_from_volume"></a>

Convert SPAMs (Spatial Probability Maps) to a maximum probability parcellation volume.

In [ ]:
########################### Testing spams2maxprob_from_volume ###########################
import clabtoolkit.imagetools as cltimg
import clabtoolkit.parcellationtools as cltparc
import numpy as np

# Five "subjects": the same 6-region parcellation shifted by a few voxels, stacked as (X, Y, Z, subjects)
parc = cltparc.Parcellation.simulate_parcellation(n_regions=6, dimensions=(40, 40, 30), seed=100)
rng = np.random.default_rng(0)
indiv_parc = np.stack(
    [np.roll(parc.data, shift=tuple(rng.integers(-2, 3, size=3)), axis=(0, 1, 2)) for _ in range(5)],
    axis=-1,
).astype(np.int32)
spams = cltimg.create_spams_from_volume(indiv_parc, np.unique(indiv_parc))

print("Test 1: Maximum probability parcellation without threshold...")
maxprob = cltimg.spams2maxprob_from_volume(spams)
print(f"  Shape: {maxprob.shape}, labels: {np.unique(maxprob)}")
print("-------------------------------")
print(" ")

print("Test 2: Keeping only the voxels where the winning region has a probability of at least 0.8...")
maxprob80 = cltimg.spams2maxprob_from_volume(spams, prob_thresh=0.8)
print(f"  Unlabeled voxels: no threshold -> {(maxprob == 0).sum()}, thresh 0.8 -> {(maxprob80 == 0).sum()}")
print("-------------------------------")
print(" ")

print("Test 3: Considering only the regions in volumes 0 and 3...")
maxprob_sub = cltimg.spams2maxprob_from_volume(spams, vol_indexes=[0, 3])
print(f"  Labels: {np.unique(maxprob_sub)}")
print(f"  The input SPAMs are not modified (probabilities still sum to 1): {np.allclose(spams.sum(axis=3), 1)}")
print("-------------------------------")


### 6.5 `compute_statistics_at_nonzero_voxels`
<a id="image-compute_statistics_at_nonzero_voxels"></a>

Compute a given statistic from a data array at the positions where the mask array is non-zero.

In [ ]:
########################### Testing compute_statistics_at_nonzero_voxels ###########################
import clabtoolkit.imagetools as cltimg
import clabtoolkit.parcellationtools as cltparc
import numpy as np

print("Test 1: Mean of a small 3D array at the non-zero positions of the mask ...")
mask = np.array([[[0, 1], [0, 0]], [[1, 0], [0, 1]]])
data = np.array([[[1, 2], [3, 4]], [[5, 6], [7, 8]]])
print(f"  Mean of [2, 5, 8]: {cltimg.compute_statistics_at_nonzero_voxels(mask, data)}")
print("-------------------------------")
print(" ")

print("Test 2: All the available metrics for one region of a parcellation over a simulated FA map...")
parc = cltparc.Parcellation.simulate_parcellation(n_regions=8, dimensions=(40, 40, 30), seed=5)
fa = cltimg.simulate_array(np.ones(parc.data.shape), distribution="uniform", low=0.2, high=0.8, random_seed=5)
region = parc.data == 3
for metric in ["mean", "std", "var", "median", "sum", "max", "min"]:
    print(f"  {metric:6s}: {cltimg.compute_statistics_at_nonzero_voxels(region, fa, metric=metric):.4f}")
print("-------------------------------")
print(" ")

print("Test 3: Mean time series of a region from a 4D image (one value per volume)...")
bold = cltimg.simulate_array(np.ones(parc.data.shape), n_volumes=6, loc=500, scale=20, random_seed=6)
ts = cltimg.compute_statistics_at_nonzero_voxels(region, bold, metric="mean")
print(f"  Time series ({ts.shape[0]} volumes): {np.round(ts, 2)}")
print("-------------------------------")
print(" ")

print("Test 4: Requesting an unsupported metric...")
try:
    cltimg.compute_statistics_at_nonzero_voxels(mask, data, metric="mode")
except ValueError as e:
    print(f"  ValueError: {e}")
print("-------------------------------")


### 6.6 `interpolate`
<a id="image-interpolate"></a>

Interpolate 3D or 4D scalar data at specified voxel coordinates using regular grid interpolation.

In [ ]:
########################### Testing interpolate ###########################
import clabtoolkit.imagetools as cltimg
import numpy as np

# Volume where every voxel stores its X index (a linear ramp along X)
ramp = np.broadcast_to(np.arange(20, dtype=float)[:, None, None], (20, 20, 20)).copy()
points = np.array([[2.0, 5.0, 5.0], [7.25, 3.0, 9.0], [12.6, 10.5, 1.2]])

print("Test 1: Linear interpolation recovers the fractional X coordinate of each point...")
print(f"  Values: {cltimg.interpolate(ramp, points)}")
print("-------------------------------")
print(" ")

print("Test 2: Nearest neighbor interpolation (useful for labels)...")
print(f"  Values: {cltimg.interpolate(ramp, points, interp_method='nearest')}")
print("-------------------------------")
print(" ")

print("Test 3: Interpolating a 4D array returns one column per volume. Points outside the volume get 0...")
ramp4d = np.stack([ramp, 2 * ramp, 3 * ramp], axis=-1)
pts = np.vstack([points, [[25.0, 5.0, 5.0]]])        # The last point is outside
print(np.round(cltimg.interpolate(ramp4d, pts), 3))
print("-------------------------------")
print(" ")

print("Test 4: Passing the coordinates with a wrong shape...")
try:
    cltimg.interpolate(ramp, np.array([1.0, 2.0, 3.0]))
except ValueError as e:
    print(f"  ValueError: {e}")
print("-------------------------------")


### 6.7 `region_growing`
<a id="image-region_growing"></a>

Fill gaps in a parcellation using a region growing algorithm.

In [ ]:
########################### Testing region_growing ###########################
import clabtoolkit.imagetools as cltimg
import clabtoolkit.parcellationtools as cltparc
import numpy as np

# Parcellation restricted to a spherical mask, with 15% of the voxels inside the mask set to 0 (gaps)
parc = cltparc.Parcellation.simulate_parcellation(n_regions=6, dimensions=(30, 30, 30), seed=11)
zz, yy, xx = np.ogrid[:30, :30, :30]
mask = (zz - 15) ** 2 + (yy - 15) ** 2 + (xx - 15) ** 2 <= 12**2
truth = (parc.data * mask).astype(np.int32)

rng = np.random.default_rng(0)
gaps = mask & (rng.random(mask.shape) < 0.15)
holed = truth.copy()
holed[gaps] = 0
print(f"Gaps introduced: {gaps.sum()} voxels")

print("Test 1: Filling the gaps with the default settings (26-neighborhood, at least 6 labeled neighbors)...")
filled = cltimg.region_growing(holed, mask)
print(f"  Gaps left: {((filled == 0) & mask).sum()}, correctly recovered: {(filled[gaps] == truth[gaps]).mean() * 100:.1f}%")
print("-------------------------------")
print(" ")

print("Test 2: A more conservative growing using the 6-neighborhood...")
filled6 = cltimg.region_growing(holed, mask, neighborhood="6", min_neighbors=4)
print(f"  Gaps left: {((filled6 == 0) & mask).sum()}, correctly recovered: {(filled6[gaps & (filled6 > 0)] == truth[gaps & (filled6 > 0)]).mean() * 100:.1f}%")
print("-------------------------------")
print(" ")

print("Test 3: Voxels outside the mask are never labeled and the input parcellation is not modified...")
print(f"  Labeled voxels outside the mask: {((filled > 0) & ~mask).sum()}")
print(f"  Input still has its gaps: {((holed == 0) & mask).sum() == gaps.sum()}")
print("-------------------------------")


### 6.8 `simulate_array`
<a id="image-simulate_array"></a>

Generate a simulated array with random values at non-zero voxel positions.

In [ ]:
########################### Testing simulate_array ###########################
import clabtoolkit.imagetools as cltimg
import numpy as np

# Sphere of radius 8 voxels centered at voxel (12, 20, 28) in a 40 x 40 x 40 volume
zz, yy, xx = np.ogrid[:40, :40, :40]
sphere = ((zz - 12) ** 2 + (yy - 20) ** 2 + (xx - 28) ** 2 <= 8**2).astype(np.uint8)

print("Test 1: Simulating a 3D array with normal values inside the sphere...")
sim = cltimg.simulate_array(sphere, loc=10, scale=2, random_seed=42)
print(f"  Shape: {sim.shape}, mean inside: {sim[sphere > 0].mean():.2f}, zeros outside: {np.all(sim[sphere == 0] == 0)}")
print("-------------------------------")
print(" ")

print("Test 2: Simulating a 4D array (8 volumes) with uniform values...")
sim4d = cltimg.simulate_array(sphere, n_volumes=8, distribution="uniform", low=-1, high=1, random_seed=42)
print(f"  Shape: {sim4d.shape}, range: [{sim4d.min():.2f}, {sim4d.max():.2f}]")
print("-------------------------------")
print(" ")

print("Test 3: The same seed gives the same values (reproducible simulations)...")
a = cltimg.simulate_array(sphere, distribution="exponential", scale=3, random_seed=7)
b = cltimg.simulate_array(sphere, distribution="exponential", scale=3, random_seed=7)
print(f"  Identical: {np.array_equal(a, b)}, mean inside: {a[sphere > 0].mean():.2f} (expected ~3)")
print("-------------------------------")
print(" ")

print("Test 4: Errors for an empty mask and invalid distribution parameters...")
try:
    cltimg.simulate_array(np.zeros((5, 5, 5)))
except RuntimeError as e:
    print(f"  RuntimeError: {e}")
try:
    cltimg.simulate_array(sphere, distribution="uniform", low=5, high=1)
except ValueError as e:
    print(f"  ValueError: {e}")
print("-------------------------------")


### 6.9 `delete_volumes_from_4D_array`
<a id="image-delete_volumes_from_4D_array"></a>

Remove specific volumes from a 4D array.

In [ ]:
########################### Testing delete_volumes_from_4D_array ###########################
import clabtoolkit.imagetools as cltimg
import numpy as np

# 4D array with 20 volumes. Each volume is filled with its own index to track which ones remain
arr = np.broadcast_to(np.arange(20, dtype=np.float32), (8, 8, 4, 20)).copy()

print("Test 1: Removing individual volumes...")
out, removed = cltimg.delete_volumes_from_4D_array(arr, vols_to_delete=[0, 1, 2, 19])
print(f"  Shape: {arr.shape} -> {out.shape}, volumes kept: {out[0, 0, 0].astype(int).tolist()}")
print("-------------------------------")
print(" ")

print("Test 2: Mixing tuples, strings with steps and numpy arrays...")
out, removed = cltimg.delete_volumes_from_4D_array(arr, vols_to_delete=[(3, 5), "10:2:16", np.array([18])])
print(f"  Removed: {removed}")
print(f"  Volumes kept: {out[0, 0, 0].astype(int).tolist()}")
print("-------------------------------")
print(" ")

print("Test 3: Errors for volumes out of range and for a 3D array...")
try:
    cltimg.delete_volumes_from_4D_array(arr, vols_to_delete=[30])
except ValueError as e:
    print(f"  ValueError: {e}")
try:
    cltimg.delete_volumes_from_4D_array(arr[..., 0], vols_to_delete=[0])
except ValueError as e:
    print(f"  ValueError: {e}")
print("-------------------------------")


### 6.10 `dilate_mm`
<a id="image-dilate_mm"></a>

Binarize a 3D array and dilate by a given distance in millimeters.

In [ ]:
########################### Testing dilate_mm ###########################
import clabtoolkit.imagetools as cltimg
import numpy as np

# A single active voxel in the center of the volume
seed = np.zeros((21, 21, 21), dtype=np.uint8)
seed[10, 10, 10] = 1

def extent(mask):
    """Number of voxels covered along each axis."""
    idx = np.argwhere(mask)
    return tuple(int(v) for v in idx.max(axis=0) - idx.min(axis=0) + 1)

print("Test 1: Dilating a single voxel by 3 mm (cube) with 1 mm and 1.5 mm isotropic voxels...")
for vs in [1.0, 1.5]:
    dil = cltimg.dilate_mm(seed, np.diag([vs, vs, vs, 1.0]), shape="cube", dilation_mm=3)
    print(f"  {vs} mm voxels: {dil.sum()} voxels, extent {extent(dil)}")
print("-------------------------------")
print(" ")

print("Test 2: Ellipsoidal dilation with anisotropic voxels (1 x 1 x 3 mm)...")
dil_sph = cltimg.dilate_mm(seed, np.diag([1.0, 1.0, 3.0, 1.0]), shape="sphere", dilation_mm=3)
print(f"  Active voxels: {dil_sph.sum()}, extent (voxels): {extent(dil_sph)}")
print("-------------------------------")
print(" ")

print("Test 3: Dilating a parcellation-like array (any non-zero value is part of the mask)...")
labels = np.zeros((30, 30, 30), dtype=np.int16)
labels[10:15, 10:15, 10:15] = 7
labels[16:20, 16:20, 16:20] = 12
dil_lab = cltimg.dilate_mm(labels, np.eye(4), dilation_mm=2)
print(f"  Non-zero voxels: {np.count_nonzero(labels)} -> {dil_lab.sum()} (output is boolean: {dil_lab.dtype})")
print("-------------------------------")
